In [1]:
import os
import time
import traceback
from datetime import datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ===============================
# 1. 유형 설정
# ===============================

property_type = 4      # 1 아파트 / 2 연립다세대 / 3 단독다가구 / 4 오피스텔 / 5 분양권 / 6 상업업무 / 7 토지 / 8 공장창고
d_type = 1             # 1 매매 / 2 전월세 --> 전월세유형: 아파트, 연립다세대, 단독다가구, 오피스텔


# ===============================
# 2. 지역 및 기간
# ===============================

region_list = [

"대전광역시","세종특별자치시","경기도","충청북도","충청남도"
]

#"서울특별시","부산광역시","대구광역시","인천광역시","광주광역시","울산광역시","전라남도","경상북도","경상남도","제주특별자치도",
"강원특별자치도","전북특별자치도"

start_date = "2021-01-01"
end_date = "2026-12-31"


# ===============================
# 3. 다운로드 폴더
# ===============================

base_dir = r"C:\startcoding\GUKTO"

type_map = {
1:"아파트",
2:"연립다세대",
3:"단독다가구",
4:"오피스텔",
5:"분양입주권",
6:"상업업무용",
7:"토지",
8:"공장창고"
}

deal_type = "매매" if d_type == 1 else "전월세"

download_dir = os.path.join(base_dir,f"{type_map[property_type]}_{deal_type}")

os.makedirs(download_dir,exist_ok=True)


# ===============================
# Chrome 설정
# ===============================

options = Options()

options.add_experimental_option("prefs",{
"download.default_directory": download_dir,
"download.prompt_for_download": False,
"download.directory_upgrade": True,
"safebrowsing.enabled": True
})


# ===============================
# 다운로드 대기
# ===============================

def wait_for_downloads(folder,timeout=60):

    for _ in range(timeout):

        if not any(f.endswith(".crdownload") for f in os.listdir(folder)):
            return

        time.sleep(1)

    print("다운로드 대기시간 초과")


# ===============================
# 처리중 팝업
# ===============================

def handle_processing_popup(driver):

    try:

        popup = WebDriverWait(driver,3).until(
            EC.visibility_of_element_located((By.CSS_SELECTOR,".layerPop"))
        )

        if "처리중입니다" in popup.text:

            popup.find_element(By.XPATH,".//button[contains(text(),'확인')]").click()

            print("팝업 확인")

    except:
        pass


# ===============================
# 날짜 범위 분리
# ===============================

def get_date_ranges(start,end):

    result=[]

    start_dt=datetime.strptime(start,"%Y-%m-%d")
    end_dt=datetime.strptime(end,"%Y-%m-%d")

    for year in range(start_dt.year,end_dt.year+1):

        from_dt=max(start_dt,datetime(year,1,1))
        to_dt=min(end_dt,datetime(year,12,31))

        result.append((year,from_dt.strftime("%Y-%m-%d"),to_dt.strftime("%Y-%m-%d")))

    return result


# ===============================
# 실행
# ===============================

driver=webdriver.Chrome(options=options)
wait=WebDriverWait(driver,30)

try:

    driver.get("https://rt.molit.go.kr/pt/xls/xls.do?mobileAt=")

    date_ranges=get_date_ranges(start_date,end_date)

    for region in region_list:

        region_folder=os.path.join(download_dir,region)
        os.makedirs(region_folder,exist_ok=True)

        for year,from_date,to_date in date_ranges:

            file_name=f"{region}_{type_map[property_type]}_{deal_type}_{year}.xlsx"
            file_path=os.path.join(region_folder,file_name)

            if os.path.exists(file_path):

                print(region,year,"이미 존재 → skip")
                continue

            try:

                wait.until(
                    EC.element_to_be_clickable((By.ID,f"xlsTab{property_type}"))
                ).click()


                # 계약일 입력

                driver.find_element(By.ID,"srhFromDt").clear()
                driver.find_element(By.ID,"srhFromDt").send_keys(from_date)

                driver.find_element(By.ID,"srhToDt").clear()
                driver.find_element(By.ID,"srhToDt").send_keys(to_date)


                # 시도 선택

                sido_select=wait.until(
                    EC.presence_of_element_located((By.ID,"srhSidoCd"))
                )

                for option in sido_select.find_elements(By.TAG_NAME,"option"):

                    if option.text.strip()==region:

                        option.click()
                        break


                # ===============================
                # 다운로드 버튼 (핵심 수정)
                # ===============================

                btn=wait.until(
                    EC.presence_of_element_located(
                        (By.XPATH,"//button[contains(text(),'EXCEL 다운')]")
                    )
                )

                driver.execute_script("arguments[0].click();",btn)

                print(region,year,"다운로드 시작")

                time.sleep(2)

                handle_processing_popup(driver)

                wait_for_downloads(download_dir)


                # 최신 파일 이동

                files = [
                    os.path.join(download_dir, f)
                    for f in os.listdir(download_dir)
                    if os.path.isfile(os.path.join(download_dir, f)) and f.endswith(".xlsx")
                ]

                newest=max(files,key=os.path.getctime)

                os.rename(newest,file_path)

                print(region,year,"저장 완료")


            except Exception as e:

                print(region,year,"오류")

                traceback.print_exc()


            time.sleep(2)


finally:

    driver.quit()

    print("수집 종료")

대전광역시 2021 다운로드 시작
대전광역시 2021 저장 완료
대전광역시 2022 다운로드 시작
대전광역시 2022 저장 완료
대전광역시 2023 다운로드 시작
대전광역시 2023 저장 완료
대전광역시 2024 다운로드 시작
대전광역시 2024 저장 완료
대전광역시 2025 다운로드 시작
대전광역시 2025 저장 완료
대전광역시 2026 다운로드 시작
대전광역시 2026 저장 완료
세종특별자치시 2021 다운로드 시작
세종특별자치시 2021 저장 완료
세종특별자치시 2022 다운로드 시작
세종특별자치시 2022 저장 완료
세종특별자치시 2023 다운로드 시작
세종특별자치시 2023 저장 완료
세종특별자치시 2024 다운로드 시작
세종특별자치시 2024 저장 완료
세종특별자치시 2025 다운로드 시작
세종특별자치시 2025 저장 완료
세종특별자치시 2026 다운로드 시작
세종특별자치시 2026 저장 완료
경기도 2021 다운로드 시작
경기도 2021 저장 완료
경기도 2022 다운로드 시작
경기도 2022 저장 완료
경기도 2023 다운로드 시작
경기도 2023 저장 완료
경기도 2024 다운로드 시작
경기도 2024 저장 완료
경기도 2025 다운로드 시작
경기도 2025 저장 완료
경기도 2026 다운로드 시작
경기도 2026 저장 완료
충청북도 2021 다운로드 시작
충청북도 2021 저장 완료
충청북도 2022 다운로드 시작
충청북도 2022 저장 완료
충청북도 2023 다운로드 시작
충청북도 2023 저장 완료
충청북도 2024 다운로드 시작
충청북도 2024 저장 완료
충청북도 2025 다운로드 시작
충청북도 2025 저장 완료
충청북도 2026 다운로드 시작
충청북도 2026 저장 완료
충청남도 2021 다운로드 시작
충청남도 2021 저장 완료
충청남도 2022 다운로드 시작
충청남도 2022 저장 완료
충청남도 2023 다운로드 시작
충청남도 2023 저장 완료
충청남도 2024 다운로드 시작
충청남도 2024 저장 완료
충청남도 2025 다운